**Inspect Models During Training**


In [ ]:
print("1")

1


In [13]:
import sys, os, torch

# make parent directory importable
sys.path.insert(0, os.path.abspath(".."))
import pretrain_and_sft_model as pm

CKPT_PATH = "/workspace/sam/minireason_archives/model_pts/astar/exp34_astar_best.pt"
ENC_CACHE_PATH = "/workspace/sam/minireason_archives/data_pts/astar/tokenisation_cache/exp34_astar_max2048_drop.pt"
SPLIT_PATH = "/workspace/sam/minireason_archives/data_pts/astar/exp34_astar_target_random_train_val_test_probe_split.pt"

N_SHOW = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# load tokenizer + checkpoint
tok = pm.Tokenizer()
ckpt = pm.load_checkpoint(CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]

# load cached tokenized dataset metadata
blob, _, ml, mi, mo, bk_counts = pm._load_enc_blob(
    ENC_CACHE_PATH,
    tok,
    safe_max_len=int(cfg.get("safe_max_len", 2048)),
    long_seq_policy=str(cfg.get("long_seq_policy", "drop")),
)

def split_trace_lines(trace: str):
    parts = [part.strip() for part in trace.split(";") if part.strip()]
    lines = []
    for part in parts:
        if part.startswith("EXPR"):
            lines.append(part)
        else:
            lines.append(part + " ;")
    return lines

ds = pm.VaryLenDataset(blob["flat_ids"], blob["offsets"], blob["lengths"], blob["bk_counts"])
split = torch.load(SPLIT_PATH, map_location="cpu")
va = torch.utils.data.Subset(ds, split["val_idx"])

# rebuild model and load weights
model_max_len = pm.infer_model_max_len_from_checkpoint(CKPT_PATH)
model = pm._build_model(
    cfg, tok, model_max_len, use_grad_ckpt=False, dropout=0.0
).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"Model params: total={sum(p.numel() for p in model.parameters()):,} | trainable={sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Validation samples: {len(va)}")
print()

for i in range(min(N_SHOW, len(va))):
    x, _, bk = va[i]
    ids = x.tolist()
    sep_idx = ids.index(tok.sep_id)

    prompt_ids = [t for t in ids[:sep_idx + 1] if t != tok.pad_id]
    target_ids = [t for t in ids[sep_idx + 1:] if t != tok.pad_id]

    gen_ids = pm.sample_one(
        model,
        tok,
        prompt_ids,
        max_out=cfg["max_output_len"],
        device=DEVICE,
        cfg=cfg,
        amp_dtype=torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16,
        use_amp=(DEVICE.type == "cuda"),
    )

    print("=" * 100)
    print(f"Example {i+1} | BACK={bk}")
    print("PROMPT:")
    print(tok.decode(prompt_ids).replace(" SEP", ""))
    print()
    print("TARGET:")
    for line in split_trace_lines(tok.decode(target_ids)):
        print(line)
    print()
    print("GENERATED:")
    for line in split_trace_lines(tok.decode(gen_ids)):
        print(line)
    print()
    print("Exact text match:", tok.decode(gen_ids).strip() == tok.decode(target_ids).strip())
    print()

/workspace/sam/minireason/pretrain_and_sft_model.py:336: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location=map_location)


/workspace/sam/minireason/pretrain_and_sft_model.py:817: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(enc_cache_path, map_location="cpu")
/tmp/ipykernel_1

Model params: total=3,429,376 | trainable=3,429,376
Loaded checkpoint: /workspace/sam/minireason_archives/model_pts/astar/exp34_astar_best.pt
Validation samples: 900

Example 1 | BACK=8
PROMPT:
IN: [ 1, 2, 2, 6, 25, 75 ] TGT: 400

TARGET:
[ 1, 2, 2, 6, 25, 75 ] -> 6 * 75 = 450 -> [ 1, 2, 2, 25, 450 ] ;
BACK -> [ 1, 2, 2, 6, 25, 75 ] ;
[ 1, 2, 2, 6, 25, 75 ] -> 6 * 25 = 150 -> [ 1, 2, 2, 75, 150 ] ;
BACK -> [ 1, 2, 2, 6, 25, 75 ] ;
[ 1, 2, 2, 6, 25, 75 ] -> 2 * 75 = 150 -> [ 1, 2, 6, 25, 150 ] ;
BACK -> [ 1, 2, 2, 6, 25, 75 ] ;
[ 1, 2, 2, 6, 25, 75 ] -> 25 + 75 = 100 -> [ 1, 2, 2, 6, 100 ] ;
[ 1, 2, 2, 6, 100 ] -> 2 * 100 = 200 -> [ 1, 2, 6, 200 ] ;
BACK -> [ 1, 2, 2, 6, 25, 75 ] ;
[ 1, 2, 2, 6, 25, 75 ] -> 6 * 75 = 450 -> [ 1, 2, 2, 25, 450 ] ;
[ 1, 2, 2, 25, 450 ] -> 450 - 25 = 425 -> [ 1, 2, 2, 425 ] ;
BACK -> [ 1, 2, 2, 6, 25, 75 ] ;
[ 1, 2, 2, 6, 25, 75 ] -> 6 * 25 = 150 -> [ 1, 2, 2, 75, 150 ] ;
[ 1, 2, 2, 75, 150 ] -> 2 * 150 = 300 -> [ 1, 2, 75, 300 ] ;
BACK -> [ 1, 2, 2, 6, 25,